# ENGR 1451/2451 — Weekly Assignment 11
**10 points**

This assignment focuses on three goals:
(i) quantifying polling uncertainty and adjusting for nonresponse;
(ii) testing specified category probabilities with a chi-square goodness-of-fit test; and
(iii) investigating categorical association while accounting for multiple tests.

*All examples and the supplied CSV are constructed classroom data, not actual surveys or measurements.*

**Due:** See Canvas.

Complete the work directly in this notebook. Show your Python code and give brief written explanations where requested. Submit the completed notebook as directed in Canvas.

## What to hand in

Work this assignment with the **course tutor** in the class project. That is the AI help to use on this assignment, and it keeps the record for you — you do not write one yourself.

At the end of each working session, type **"make my record"** in the chat. Copy the block it produces and paste it into the **Session records** cell near the bottom of this notebook.

You do not have to finish in one sitting. To pick up later, start a new chat in the same project, say which assignment you are on and which problem you stopped at, and keep going. That session gives you its own block, which you paste under the previous one.

Records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. The code and the written answers still have to be yours.

## LE11-1. Polling: precision and nonresponse (3 pts)

A city is considering extending its public library hours. Let $p$ be the proportion of **all adult residents** who support the proposal.

### A. Calculate poll estimates (2 pts)

**Survey 1: a simple random sample.** Researchers randomly select **900 different adults** from a complete list of adult residents. Everyone selected responds, and **513 support** the proposal. The population is large enough to ignore the finite population correction.

Using Python:

1. Calculate the sample proportion $\hat p$.
2. Since $p$ is unknown, use $\hat p$ to estimate it and calculate the **estimated standard error**:

$$
\widehat{\mathrm{SE}}(\hat p)=\sqrt{\frac{\hat p(1-\hat p)}{n}}.
$$

   Calculate this estimated standard error, the approximate 95% margin of error $2\times\widehat{\mathrm{SE}}(\hat p)$, and the interval $\hat p\pm 2\widehat{\mathrm{SE}}(\hat p)$. Use the lecture's multiplier **2**. Report support and interval endpoints as percentages and the margin as **percentage points**, to one decimal place.

**Survey 2: a separate survey with nonresponse.** A second random invitation sample produces the responses in `responses` below. Residents living **within 3 km** of the library are 40% of the population; those living **farther away** are 60%. These population shares are known, but the respondent shares differ.

3. Calculate each group's support proportion and the **unweighted** proportion supporting the proposal among all respondents.
4. Calculate a **weighted** population estimate using

$$
\hat p_{\mathrm{weighted}}=\sum_g w_g\hat p_g,
$$

   where $w_g$ is the known **population share**, not the share of respondents, and $\hat p_g$ is the support proportion among respondents in group $g$. Display the group proportions and both overall estimates as percentages. For Survey 2, calculate **point estimates only**.

In [276]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 4)
alpha = 0.05

# Survey 1: complete responses to a simple random sample.
n = 900
support = 513

# Survey 2: separate responses; population shares are supplied.
responses = pd.DataFrame({
    "area": ["Within 3 km", "Farther away"],
    "population_share": [0.40, 0.60],
    "respondents": [600, 300],
    "support": [420, 120],
})
display(responses)

# Your code here
# Survey 1: calculate p_hat, the estimated SE, margin, and interval.
p_hat = support/n
se_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)
margin_of_error = 2*se_estimated
interval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])

print("p_hat [%] =",round(p_hat*100,1),
    "\nEstimated SE [%] =",round(se_estimated*100,1),
    "\nMargin of Error [% points] =",round(margin_of_error*100,1),
    "\n95% confidence interval for percentage of adults supporting the library hours proposal:",(interval*100).round(1))

# Survey 2: calculate group support rates and unweighted/weighted estimates.
support_rate_unweighted = sum(responses["support"])/sum(responses["respondents"])

support_rates_group = responses["support"]/responses["respondents"]
support_rates_weighted = support_rates_group*responses["population_share"]
support_rate_weighted = sum(support_rates_weighted)

support_rates_group.index = responses["area"]
print("\nUnweighted proportion [%]:",round(support_rate_unweighted*100,2),
      "\nWeighted proportion [%]:",round(support_rate_weighted*100,2))
print("\nGroup proportion [%]:\n",support_rates_group*100)


,area,population_share,respondents,support
0,Within 3 km,0.4,600,420
1,Farther away,0.6,300,120


p_hat [%] = 57.0 
Estimated SE [%] = 1.7 
Margin of Error [% points] = 3.3 
95% confidence interval for percentage of adults supporting the library hours proposal: [53.7 60.3]

Unweighted proportion [%]: 60.0 
Weighted proportion [%]: 52.0

Group proportion [%]:
 area
Within 3 km     70.0
Farther away    40.0
dtype: float64


### B. Interpret the surveys (1 pt)

Answer in about **3–4 sentences**:

1. For Survey 1, explain what “approximately 95% confidence” means in terms of **repeating the sampling procedure**.
2. For Survey 2, explain the direction of the change from the unweighted to the weighted estimate. Which group was overrepresented among respondents?
3. What assumption about respondents and nonrespondents **within each area** is needed for weighting to remove the nonresponse bias? Would the ordinary random-sampling margin of error account for a violation of that assumption?

**Answer:**
1. “Approximately 95% confidence” means that if we repeated our sampling procedure, we would expect 95% of confidence intervals constructed in such a manner to contain the true population proportion of people who support the library hour proposal.
2. The direction of change from unweighted to weighted was negative--i.e. the proportion estimate decreased. This change is direction is due to the "Within 3 km" group being overrepresented among the respondents (they had a larger share of respondents despite having a smaller population share).
3. Assuming that the nonrespondents would have the same rates of support for the proposal as those who responded is neccesary in order for weighting to remove bias. The ordinary random-sampling margin of error would not be able to account for any violation of this assumption because it does not take into account the fact that the results are biased in its formula. Bias is a systematic error that can not be correctd by a margin of error.

## LE11-2. Does a router follow its configured probabilities? (3 pts)

A router is configured to assign each packet **independently at random** to servers A, B, C, and D with probabilities **0.40, 0.30, 0.20, and 0.10**, respectively. These probabilities were fixed **before** the packet counts were observed; they are not estimated from the data. A test of **400 packets** produces counts **183, 112, 79, and 26**.

### A. Chi-square goodness of fit (2 pts)

Using Python:

1. Calculate the expected count for each server, $E_i=n p_i$. Use the configured probabilities; the servers are **not equally likely**.
2. Display a table containing server, observed count, expected count, and chi-square contribution $(O_i-E_i)^2/E_i$. Sum the contributions to obtain the test statistic.
3. Determine the degrees of freedom. Check the lecture's rule of thumb that **every expected count is at least 5**.
4. Run `stats.chisquare(f_obs=observed, f_exp=expected)`. Confirm that its statistic matches your manual calculation. Print $\chi^2$, degrees of freedom, the **right-tail $P$-value**, and the decision at $\alpha=0.05$.

Report $\chi^2$ to **three decimals** and $P$ to **four decimals**. Use unrounded quantities in calculations.

In [277]:
servers = ["A", "B", "C", "D"]
probabilities = np.array([0.40, 0.30, 0.20, 0.10])
observed = np.array([183, 112, 79, 26])

# Your code here
# Calculate expected counts using observed.sum() and probabilities.
expected = probabilities*observed.sum()

# Build the contributions table and calculate chi-square manually.
contributions = ((observed-expected)**2)/expected
chi_statistic = sum(contributions)
print("Chi-square statistic (manual) =",round(chi_statistic,3))

chi_table = pd.DataFrame({
    "Server":servers,
    "Observed counts":observed,
    "Expected counts":expected,
    "Chi-square contribution":contributions,
})

# Run stats.chisquare(...), report df and P, and check expected counts.
alpha = 0.05
chi_test = stats.chisquare(f_obs=observed,f_exp=expected)

decision = "Fail to Reject H0"
if(chi_test.pvalue < alpha):
    decision = "Reject H0"

warning_status = "statistified"
for item in expected:
    if(item < 5):
        warning_status = "VIOLATED"

equality_status = np.isclose(chi_statistic, chi_test.statistic)

print("Chi-square statistic (stats.chisquare(...)) =",round(chi_test.statistic,3), "\tMatches manual? -->",equality_status,
      "\ndf =",len(probabilities)-1,
      "\nP-value =",round(chi_test.pvalue,4),
      "\nDecision at alpha =",alpha,"-->",decision,
      "\nAll expected counts >= 5? -->",warning_status)


chi_table.head()

Chi-square statistic (manual) = 8.752
Chi-square statistic (stats.chisquare(...)) = 8.752 	Matches manual? --> True 
df = 3 
P-value = 0.0328 
Decision at alpha = 0.05 --> Reject H0 
All expected counts >= 5? --> statistified


,Server,Observed counts,Expected counts,Chi-square contribution
0,A,183,160.0,3.3062
1,B,112,120.0,0.5333
2,C,79,80.0,0.0125
3,D,26,40.0,4.9000


### B. Interpret the discrepancy (1 pt)

Answer in about **3–4 sentences**:

1. State $H_0$ and $H_A$ in terms of the server-assignment probabilities, and give your conclusion at the 5% level.
2. Which server contributes most to $\chi^2$? Is it overrepresented or underrepresented relative to its expected count?
3. Explain the $P$-value as a probability **assuming $H_0$ and independent assignments**. Does it give the probability that the router's configured probabilities are correct?

**Answer:**
1. $H_0$: The router assigns packets to servers A, B, C, and D according to their expected probabilities of 0.40, 0.30, 0.20, and 0.10, respectively. $H_A$: The router does not assign packets to the servers according to said expected probabilities. Conclusion at $\alpha = 0.05$: since $P < \alpha$, we reject $H_0$. We have statistically significant evidence that the router does not assign packets to the servers according to said expected probabilities.
2. Server D contributes the most to $\chi^2$. It is underrepresented relative to its expected count.
3. The p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme (greater) as the one obtained in the sample of independent packets under $H_0$ (meaning the router is expected to truly asssign packets according to the expected probabilities). It does not provide the probability that the router's configured probabilities are correct.

## LE11-3. Employee travel: association and multiple tests (4 pts)

### A. Analyze individual records from a CSV (2.5 pts)

The supplied file `./data/LE11_employee_commutes.csv` contains **300 constructed employee records**. Each row is one employee, with three columns:

| Column | Meaning |
|---|---|
| `employee_id` | Unique anonymous classroom identifier |
| `shift` | Day, Evening, or Night |
| `commute_mode` | Car, Public transit, or Walk or bike |

In this hypothetical study, employees are sampled at random from a large workforce, each responds once, and there is no nonresponse. Treat different employees' responses as independent. **Shifts were not randomly assigned.**

We want to know whether **shift and usual commute mode are associated** in the workforce. Build the table from the individual records; the contingency table is not supplied.

In [278]:
commutes = pd.read_csv("./data/employee_commutes.csv")
display(commutes.head())
print(f"Employees: {len(commutes)}")
print(f"Unique IDs: {commutes['employee_id'].nunique()}")

,employee_id,shift,commute_mode
0,E001,Evening,Car
1,E002,Day,Walk or bike
2,E003,Evening,Public transit
3,E004,Day,Walk or bike
4,E005,Day,Walk or bike


Employees: 300
Unique IDs: 300


Use Python to:

1. Build an observed-count table using `pd.crosstab()`, with **shift as rows** and **commute mode as columns**. Also display percentages **within each shift**; every percentage row should sum to 100%.
2. Calculate the expected count for **Day × Public transit** by hand from the table margins:

$$
E_{ij}=\frac{(\text{row total})(\text{column total})}{N}.
$$

3. Run `commute_test = stats.chi2_contingency(observed_table, correction=False)` on the **counts**. Display a labeled table of all expected counts, check the smallest expected count, and confirm the expected count from step 2.
4. Report $\chi^2$, $df=(r-1)(c-1)$, $P$, and the decision using the **unadjusted** $\alpha=0.05$ cutoff. Use three decimals for $\chi^2$ and four for $P$.

In **3–4 sentences**, state $H_0$ and $H_A$, describe the pattern in the within-shift percentages, and interpret the test. Would this study establish that changing an employee's shift **causes** a change in commute mode?

**Answer:** $H_0$: shift and mode of transportation are independent. $H_A$: the shift and mode of transportation distributions differ. This dataset shows a decreasing percentage of commuters opting for public transit / walk or bike as the shift goes from day --> evening --> night. Given that the p-value is less than $\alpha = 0.05$, we reject $H_0$--we have statistically significant evidence that the shift and mode of transportation distributions differ from one another. This study does not establish that changing an empoyee's shift causes a change in commute mode, as the study did not randomly assign employees to shifts, preventing it from controlling for cofounding variables, such as commute preference or distance.

In [279]:
# Your code here

# Build observed_table using pd.crosstab(...).
observed_table = pd.crosstab(commutes["shift"],commutes["commute_mode"])

# Calculate percentages within each shift (not percentages of all employees).
percentage_table = pd.crosstab(commutes["shift"],commutes["commute_mode"],normalize = "index")*100

# Calculate the Day x Public transit expected count from the margins.
expected_count_ij = (sum(observed_table.loc["Day"])*(observed_table["Public transit"].sum()))/(sum(observed_table.loc["Day"]) + sum(observed_table.loc["Evening"]) + sum(observed_table.loc["Night"]))

# Run the chi-square test and keep the result in commute_test.
commute_test = stats.chi2_contingency(observed_table,correction=False)
expected_count_table = pd.DataFrame([
    commute_test.expected_freq[0,:],
    commute_test.expected_freq[1,:],
    commute_test.expected_freq[2,:],
],index=observed_table.index,
columns=observed_table.columns)

observed_table.head()


commute_mode,Car,Public transit,Walk or bike
shift,,,
Day,70,50,30
Evening,48,26,16
Night,44,9,7


In [280]:
#check step
print((percentage_table).loc["Day"].sum(),
      (percentage_table).loc["Evening"].sum(), 
      (percentage_table).loc["Night"].sum())

#Printing
(percentage_table).head()

100.0 100.0 100.0


commute_mode,Car,Public transit,Walk or bike
shift,,,
Day,46.6667,33.3333,20.0000
Evening,53.3333,28.8889,17.7778
Night,73.3333,15.0000,11.6667


In [281]:
# Display expected counts; report chi-square, df, P, and the decision.
decision = "Fail to Reject H0"
if(commute_test.pvalue < alpha):
    decision = "Reject H0"

warning_status = "statistified"
if((commute_test.expected_freq).min()<5):
    warning_status = "VIOLATED"

print("Chi-square =",round(commute_test.statistic,3),
      "\ndf =",((len(observed_table["Car"])-1)*(len(observed_table.loc["Day"])-1)),
      "\nP-value =",round(commute_test.pvalue,4),
      "\nDecision at alpha =",alpha,"-->",decision,
      "\nAll expected counts >= 5? -->",warning_status)

print("Expected count for public transit during the day =",expected_count_ij,"\tEqual to respective value? -->",np.isclose(expected_count_ij, expected_count_table["Public transit"].loc["Day"]))
expected_count_table.head()

Chi-square = 12.438 
df = 4 
P-value = 0.0144 
Decision at alpha = 0.05 --> Reject H0 
All expected counts >= 5? --> statistified
Expected count for public transit during the day = 42.5 	Equal to respective value? --> True


commute_mode,Car,Public transit,Walk or bike
shift,,,
Day,81.0,42.5,26.5
Evening,48.6,25.5,15.9
Night,32.4,17.0,10.6


### B. Account for the full family of tests (1.5 pts)

The team explored **eight** possible factors associated with commute mode in the same survey. The shift test in part A is one of them. The other seven **unadjusted** $P$-values are supplied below; no additional datasets or tests are needed. Treat **all eight** tests as one family, regardless of their results. These are hypothetical results from valid individual tests.

1. Add the **unrounded** shift $P$-value from `commute_test.pvalue` to the seven supplied results.
2. Compute the **Bonferroni cutoff** $\alpha_{\mathrm{each}}=0.05/8$. Display a table showing which of the eight results are significant at the ordinary 0.05 cutoff and which remain significant at the Bonferroni cutoff.
3. In **2–3 sentences**, identify which conclusions change, including the shift result. Explain why reporting only the ordinary $P<0.05$ results would misrepresent the search, and whether a result that fails the adjusted cutoff proves there is no association.

**Answer:**
The conclusions that change from being significant at $\alpha = 0.05$ to no longer being significant at $\alpha_{\mathrm{each}} = 0.00625$ are shift, commute distance, and flexible start time (parking access and houshold vehicles remain significant). The ordinary $P<0.05$ results would misrepresent the search because it would hide from the reader the fact that, under $H_0$, if we run a large amount of significane tests, we are likley to find a few p-values that are less than 0.05 (this would mislead the read into thinking the results are truly significant and not just due to random chance from such a large number of tests). If a result fails the adjusted cutoff, it does not prove there is no association, rather, it does not provide evidence in support of an association (there may still be a weak association that the test was unable to pick up on).

In [282]:
other_tests = pd.DataFrame({
    "factor": [
        "Parking access", "Household vehicles", "Commute distance",
        "Flexible start time", "Tenure", "Department", "Job category",
    ],
    "p_value": [0.0012, 0.0041, 0.0170, 0.0410, 0.1800, 0.4600, 0.7300],
})

# Use unrounded P-values for comparisons; round only for display.


# Your code here

# Add the shift result from part A to form a table of all eight tests.
all_tests = pd.concat([other_tests,pd.DataFrame({"factor":["Shift"],"p_value":[commute_test.pvalue]})],ignore_index=True)

all_tests.head(9)

,factor,p_value
0,Parking access,0.0012
1,Household vehicles,0.0041
2,Commute distance,0.0170
3,Flexible start time,0.0410
4,Tenure,0.1800
5,Department,0.4600
6,Job category,0.7300
7,Shift,0.0144


In [283]:
# Calculate the Bonferroni cutoff and add both significance indicators.
alpha_each = alpha/len(all_tests)
all_tests["Intially significant at alpha = 0.05?"] = (all_tests["p_value"] < alpha)
all_tests["Still significant and alpha_each?"] = (all_tests["p_value"] < alpha_each)

print("Boneferroni cutoff (alpha_each):",alpha_each)

all_tests.sort_values("p_value").head(9)

Boneferroni cutoff (alpha_each): 0.00625


,factor,p_value,Intially significant at alpha = 0.05?,Still significant and alpha_each?
0,Parking access,0.0012,True,True
1,Household vehicles,0.0041,True,True
7,Shift,0.0144,True,False
2,Commute distance,0.0170,True,False
3,Flexible start time,0.0410,True,False
4,Tenure,0.1800,False,False
5,Department,0.4600,False,False
6,Job category,0.7300,False,False


## Session records

At the end of each working session, type **"make my record"** in the chat and paste the block it produces into the cell below. If you work across several sessions, paste each new block underneath the previous one.

Session records are not graded for correctness or for how much help you needed, and extra credit is given for a record that shows you attempted the problem. They are read to find out which steps are hardest, and used to improve the course.

<!-- Paste your session record blocks in this cell, one after another. -->

*(session records go here)*
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA11",
  "export_n": 1,
  "session_date": "2026-11-15",
  "messages": [
    {"i": 1, "who": "student", "text": "How can I round the numbers displayed in a tuple?"},
    {"i": 2, "who": "tutor", "text": "Good question, and it comes up a lot with SciPy results. A tuple holds its values at full precision, and `print` shows all of them, so you format at the moment of display. [trimmed] (round() on each element, f-strings, formatting every element; round() changes stored value, f-strings do not; tuples are immutable; unpacking; pd.set_option only affects pandas display.)"},
    {"i": 3, "who": "student", "text": "Would it be easier to round a numpy array?"},
    {"i": 4, "who": "tutor", "text": "Often, yes. NumPy has a built-in for it, so you do not need a loop or a generator expression the way you do with a tuple. [trimmed] (np.round / .round, one setting for the whole array, np.set_printoptions, f-string formatting, pandas .round with a dict.)"},
    {"i": 5, "who": "student", "text": "How can I rename new columns during pd.concat"},
    {"i": 6, "who": "tutor", "text": "`pd.concat` does not rename columns itself. It matches columns **by name**, so what you rename is the thing you hand to it. [trimmed] (rename before/inside/after concat, mismatched names give NaN, ignore_index, keys=, axis=1, add_suffix, DataFrame.append removed in pandas 2.0.)"},
    {"i": 7, "who": "student", "text": "Did I miss anything for 1A?"},
    {"i": 8, "who": "tutor", "text": "I can check it, but I can't see your 1A cell yet. Could you paste the cell you've been working on, along with what it prints? [trimmed] (checklist of 1A prompt items.)"},
    {"i": 9, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"Margin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rates_unweighted = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_unweighted*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_unweighted,support_rates_weighted],axis=1,keys=[\"unweighted\",\"weighted\"])\ndisplay_table.head()\n\n```"},
    {"i": 10, "who": "tutor", "text": "Your Survey 1 logic is sound. [trimmed] (p̂ and SE not printed; group rates mislabeled unweighted; weighted terms never summed; Survey 2 not shown as percentages; row labels 0 and 1.)"},
    {"i": 11, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"Margin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_group,support_rates_weighted],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",support_rate_unweighted)\ndisplay_table.head()\n\n```"},
    {"i": 12, "who": "tutor", "text": "You fixed the unweighted estimate: `sum(responses[\"support\"]) / sum(responses[\"respondents\"])` is the right construction. [trimmed] (p̂ and SE still not printed; weighted still two terms; percent display; row labels.)"},
    {"i": 13, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1)\n    \"\\nMargin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_group,support_rates_weighted],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",support_rate_unweighted)\ndisplay_table.head()\n\n```"},
    {"i": 14, "who": "tutor", "text": "Nice progress: p̂ and the estimated SE now have their own printed lines. **This version will not run.** [trimmed] (missing comma after round(se_estimated*100,1) causes SyntaxError; margin label, weighted estimate, percent display, row labels still open.)"},
    {"i": 15, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1),\n    \"\\nMargin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_group*100,support_rates_weighted*100],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2))\ndisplay_table.head()\n\n```"},
    {"i": 16, "who": "tutor", "text": "The missing comma is fixed, so the cell should run now, and you've started converting Survey 2 to percentages. [trimmed] (margin label, weighted estimate still two terms, row labels, unweighted display label.)"},
    {"i": 17, "who": "student", "code": true, "text": "How do you want percentage points to be done for the margin  of error? The units are already denotated as percentage and not proportion.\n\n```python\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\n#support_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rate_unweighted*responses[\"population_share\"]\n\n#display_table = pd.concat([support_rates_group*100,support_rates_weighted*100],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2))\n#display_table.head()\n```"},
    {"i": 18, "who": "tutor", "text": "Your number is already right: `round(margin_of_error*100, 1)` gives the margin on the percentage scale, and that is exactly what \"percentage points\" means. The only change is the label. [trimmed] (percent vs percentage points; the weighted line now multiplies the overall unweighted rate by the shares instead of the group rates; generic .sum() example.)"},
    {"i": 19, "who": "student", "text": "What, specifically, was the issue with my response to survey 2?"},
    {"i": 20, "who": "tutor", "text": "There are three separate issues, and I'll name the exact lines. [trimmed] (wrong ingredient in weighted line; weighted estimate never reduced to one number; percentage display and row labels.)"},
    {"i": 21, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1),\n    \"\\nMargin of Error [% points] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\nsupport_rate_weighted = sum(support_rates_weighted)\n\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2),\n      \"\\nWighted proportion:\",round(support_rate_weighted*100,2))\n\n\n```"},
    {"i": 22, "who": "tutor", "text": "Both missing pieces are in now: the weighted estimate is a single number, `sum(support_rates_weighted)`, and the margin is labeled in percentage points. [trimmed] (group proportions not displayed; \"Wighted\" typo; naming note.)"},
    {"i": 23, "who": "student", "code": true, "text": "```python\nprint(support_rates_group.set_index(responses[\"area\"]))\n```\n\n---------------------------------------------------------------------------\nAttributeError                            Traceback (most recent call last)\nCell In[12], line 41\n     37 support_rate_weighted = sum(support_rates_weighted)\n     38 \n     39 print(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2),\n     40       \"\\nWighted proportion:\",round(support_rate_weighted*100,2))\n---> 41 print(support_rates_group.set_index(responses[\"area\"]))\n\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\generic.py:6194, in NDFrame.__getattr__(self, name)\n   6190             and name not in self._accessors\n   6191             and self._info_axis._can_hold_identifiers_and_holds_name(name)\n   6192         ):\n   6193             return self[name]\n-> [6194](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/generic.py:6194)         return object.__getattribute__(self, name)\n\nAttributeError: 'Series' object has no attribute 'set_index'"},
    {"i": 24, "who": "tutor", "text": "The error message is telling you exactly what's wrong: `support_rates_group` is a **Series** (one column of numbers), and `.set_index` is a **DataFrame** method. [trimmed] (three options for labeling a Series, alignment caution, percent formatting.)"},
    {"i": 25, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1),\n    \"\\nMargin of Error [% points] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\n\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\nsupport_rate_weighted = sum(support_rates_weighted)\n\nsupport_rates_group.index = responses[\"area\"]\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2),\n      \"\\nWighted proportion:\",round(support_rate_weighted*100,2))\nprint(\"\\nGroup proportions:\\n\",support_rates_group)\n\n```"},
    {"i": 26, "who": "tutor", "text": "Your group proportions now carry the area names. [trimmed] (order-of-assignment alignment caution; group proportions still not percentages; typo; overall-line labels.)"},
    {"i": 27, "who": "student", "code": true, "text": "```python\nsupport_rates_group.index = responses[\"area\"]\nprint(\"\\nUnweighted proportion [%]:\",round(support_rate_unweighted*100,2),\n      \"\\nWighted proportion [%]:\",round(support_rate_weighted*100,2))\nprint(\"\\nGroup proportion [%]:\\n\",support_rates_group*100)\n```\n\ndone"},
    {"i": 28, "who": "tutor", "text": "Those lines cover what the prompt asks: the group proportions now print as percentages with area names, and both overall estimates carry the `[%]` label. [trimmed] (typo note; Restart and Run All; offered 1B or record.)"},
    {"i": 29, "who": "student", "text": "```markdown\n**Answer:**\n1. “Approximately 95% confidence” means that if we repeated our sampling procedure, we would expect 95% of confidence intervals constructed in such a manner to contain the true population proportion of people who support the library hour proposal.\n2. The direction of change from weighted to unweighted was negative--i.e. the proportion estimate decreased. The \"Within 3 km\" group was overrepresented among the respondents because they had a larger share of respondents despite having a smaller population share.\n3. Assuming that nonresponse rates are equal for both areas is neccesary in order for weighting to remove the nonresponse bias. The ordinary random-sampling margin of error would not be able to account for any violation of this assumption because it does not take into account the fact that the results are biased in its formula. \n```"},
    {"i": 30, "who": "tutor", "text": "Thanks for posting a draft. Per the prompt, your answer needs to be graded as your own writing, so I'll point at where the reasoning holds and where it needs another look, without rewriting anything. [trimmed] (Q1 solid; Q2 direction wording and reason; Q3 assumption is about people within one area, not equal response rates; length.)"},
    {"i": 31, "who": "student", "text": "```markdown\n2. The direction of change from unweighted to weighted was negative--i.e. the proportion estimate decreased. This change is direction is due to the \"Within 3 km\" group being overrepresented among the respondents (they had a larger share of respondents despite having a smaller population share).\n3. Assuming that the nonrespondents would have the rates of support for the proposal is neccesary in order for weighting to remove bias. The ordinary random-sampling margin of error would not be able to account for any violation of this assumption because it does not take into account the fact that the results are biased in its formula. Bias is a systematic error that can not be correctd by a margin of error.\n```"},
    {"i": 32, "who": "tutor", "text": "Your revision is closer, and the direction wording in question 2 now matches the prompt. [trimmed] (Q2 needs the link to the group support rates; Q3 sentence incomplete; margin-of-error reasoning strong; length.)"},
    {"i": 33, "who": "student", "text": "```markdown\n3. Assuming that the nonrespondents would have the same rates of support for the proposal is neccesary in order for weighting to remove bias.\n```"},
    {"i": 34, "who": "tutor", "text": "You've got the core of it. The assumption is about equal support rates, and you now say it is what weighting needs. [trimmed] (name the comparison group and that it holds within each area; typo; assemble all three answers.)"},
    {"i": 35, "who": "student", "text": "```markdown\nAssuming that the nonrespondents would have the same rates of support for the proposal as those who responded is neccesary in order for weighting to remove bias.\n```"},
    {"i": 36, "who": "tutor", "text": "That is the missing piece: you now name the comparison, nonrespondents against respondents. [trimmed] (within each area still not stated; typo; assemble; LE11-2 next.)"},
    {"i": 37, "who": "student", "code": true, "text": "```python\nservers = [\"A\", \"B\", \"C\", \"D\"]\nprobabilities = np.array([0.40, 0.30, 0.20, 0.10])\nobserved = np.array([183, 112, 79, 26])\n\n# Your code here\n# Calculate expected counts using observed.sum() and probabilities.\nexpected = probabilities*observed.sum()\n\n# Build the contributions table and calculate chi-square manually.\ncontributions = ((observed-expected)**2)/expected\nchi_statistic = sum(contributions)\nprint(\"Chi-square statistic (manual) =\",round(chi_statistic,3))\n\nchi_table = pd.DataFrame({\n    \"Server\":servers,\n    \"Observed counts\":observed,\n    \"Expected counts\":expected,\n    \"Chi-square contribution\":contributions,\n})\n\n# Run stats.chisquare(...), report df and P, and check expected counts.\nalpha = 0.05\nchi_test = stats.chisquare(f_obs=observed,f_exp=expected)\n\ndecision = \"Fail to Reject H0\"\nif(chi_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nwarning_status = \"statistified\"\nfor item in expected:\n    if(item <= 5):\n        warning_status = \"VIOLATED\"\n\nprint(\"Chi-square statistic (stats.chisquare(...)) =\",round(chi_test.statistic,3),\n      \"\\ndf =\",len(probabilities)-1,\n      \"\\nP-value =\",round(chi_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n\n\nchi_table.head()\n```"},
    {"i": 38, "who": "tutor", "text": "Your LE11-2A cell covers each item the prompt lists, and the logic is sound. [trimmed] (rule-of-thumb should be < 5 not <= 5; explicit match check with np.isclose; typo; keep alpha unchanged later.)"},
    {"i": 39, "who": "student", "code": true, "text": "```python\nwarning_status = \"statistified\"\nfor item in expected:\n    if(item < 5):\n        warning_status = \"VIOLATED\"\n\nequality_status = np.isclose(chi_statistic, chi_test.statistic)\n\nprint(\"Chi-square statistic (stats.chisquare(...)) =\",round(chi_test.statistic,3), \"\\tMatches manual? -->\",equality_status,\n      \"\\ndf =\",len(probabilities)-1,\n      \"\\nP-value =\",round(chi_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n```"},
    {"i": 40, "who": "tutor", "text": "Both changes are in. `item < 5` now matches \"at least 5\", and `np.isclose(...)` compares the unrounded values. That covers every item on the LE11-2A list. [trimmed] (label typo; Restart and Run All; listed the three LE11-2B questions.)"},
    {"i": 41, "who": "student", "text": "```markdown\n**Answer:**\n1. $H_0$: The router assigns packets to routers A, B, C, and D according to their expected probabilities of 0.40, 0.30, 0.20, and 0.10, respectively. $H_A$: The router does not assign packets to the routers according to said expected probabilities.\n2. Server D contributs the most to $\\chi^2$. It is underrepresented relative to its expected count.\n3. The p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme as the one obtained in the sample under $H_0$. It does not provide the probability that the router's configured probabilities are correct.\n```"},
    {"i": 42, "who": "tutor", "text": "Your draft is in good shape, and the core ideas are right. [trimmed] (Q1 missing the 5% conclusion, servers not routers; Q2 consistent with reference; Q3 missing independence and tail check; length; typo.)"},
    {"i": 43, "who": "student", "text": "```markdown\n1. $H_0$: The router assigns packets to routers A, B, C, and D according to their expected probabilities of 0.40, 0.30, 0.20, and 0.10, respectively. $H_A$: The router does not assign packets to the routers according to said expected probabilities. Conclusion at $\\alpha = 0.05$: since $P < \\alpha$, we reject $H_0$. We have statistically significant evidence that the router does not assign packets to the routers according to said expected probabilities.\n2. Server D contributes the most to $\\chi^2$. It is underrepresented relative to its expected count.\n3. The p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme (greater) as the one obtained in the sample under $H_0$ (meaning the servers are expected to truly asssign packets according to the expected probabilities). It does not provide the probability that the router's configured probabilities are correct.\n```"},
    {"i": 44, "who": "tutor", "text": "Question 1 is now complete: the hypotheses are about the assignment probabilities, and your conclusion matches your own P and α. [trimmed] (servers wording; Q3 independence and repeated-test framing; typos; length.)"},
    {"i": 45, "who": "student", "text": "```markdown\n3. The p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme (greater) as the one obtained in the sample under $H_0$ (meaning the router is expected to truly asssign packets according to the expected probabilities). It does not provide the probability that the router's configured probabilities are correct.\n```"},
    {"i": 46, "who": "tutor", "text": "You're close on question 3. [trimmed] (independence still missing; what gets repeated; typos; offered a different-example walkthrough.)"},
    {"i": 47, "who": "student", "text": "```markdown\nThe p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme (greater) as the one obtained in the sample of independent packets under $H_0$ (meaning the router is expected to truly asssign packets according to the expected probabilities).\n```"},
    {"i": 48, "who": "tutor", "text": "Independence is in now, and your sentence says the packets are independent and that the calculation is done under H₀. [trimmed] (dropped the not-the-probability-configured-is-correct sentence; typos; asked about LE11-3 or record.)"},
    {"i": 49, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.concat([observed_table.loc[observed_table[\"shift\"]==\"Day\"]/sum(observed_table.loc[observed_table[\"shift\"]==\"Day\"]),\n                              observed_table.loc[observed_table[\"shift\"]==\"Evening\"]/sum(observed_table.loc[observed_table[\"shift\"]==\"Evening\"]),\n                              observed_table.loc[observed_table[\"shift\"]==\"Night\"]/sum(observed_table.loc[observed_table[\"shift\"]==\"Night\"]),])\n\n# Calculate the Day x Public transit expected count from the margins.\n# Run the chi-square test and keep the result in commute_test.\n# Display expected counts; report chi-square, df, P, and the decision.\n\nobserved_table.head()\n\n```\n\n---------------------------------------------------------------------------\nKeyError                                  Traceback (most recent call last)\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\indexes\\base.py:3641, in Index.get_loc(self, key)\n   3640 try:\n-> [3641](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/indexes/base.py:3641)     return self._engine.get_loc(casted_key)\n   3642 except KeyError as err:\n\nKeyError: 'shift'\n\nThe above exception was the direct cause of the following exception:\n\nKeyError                                  Traceback (most recent call last)\nCell In[52], line 7\n...\nKeyError: 'shift'\nOutput is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings..."},
    {"i": 50, "who": "tutor", "text": "Your `pd.crosstab(commutes[\"shift\"], commutes[\"commute_mode\"])` line is right, and the bracket access to `shift` is exactly what it needs. The error comes from the next statement. [trimmed] (after crosstab, shift is the row index not a column; .loc[\"Day\"]; normalize=\"index\"; percentages need *100.)"},
    {"i": 51, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_total_ij = (sum(observed_table.loc[\"Day\"])*(sum(observed_table.loc[\"Car\"])))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\n\n\n# Display expected counts; report chi-square, df, P, and the decision.\n\n\n\nobserved_table.head()\n```\n\n---------------------------------------------------------------------------\nKeyError                                  Traceback (most recent call last)\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\indexes\\base.py:3641, in Index.get_loc(self, key)\n   3640 try:\n-> [3641](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/indexes/base.py:3641)     return self._engine.get_loc(casted_key)\n   3642 except KeyError as err:\n\nKeyError: 'Car'\n\nThe above exception was the direct cause of the following exception:\n\nKeyError                                  Traceback (most recent call last)\nCell In[79], line 10\n...\nKeyError: 'Car'"},
    {"i": 52, "who": "tutor", "text": "Your crosstab lines are right now. `normalize = \"index\"` is the setting that divides each row by its own row total. The error is on the expected-count line. [trimmed] (.loc takes row labels; columns use brackets; wrong column Car vs Public transit; N alternatives; percentages need *100 and row-sum check.)"},
    {"i": 53, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_count_ij = (sum(observed_table.loc[\"Day\"])*(observed_table[\"Public transit\"].sum()))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\ncommute_test = stats.chi2_contingency(observed_table,correction=False)\nexpected_count_table = pd.DataFrame({\n    \"Car\":commute_test.expected_freq[:,0],\n    \"Public transit\":commute_test.expected_freq[:,1],\n    \"Walk or bike\":commute_test.expected_freq[:,2],\n},index=[\"Day\",\"Evening\",\"Night\"])\n\nobserved_table.head()\n```\n\n\n```python\npercentage_table.head()\n```\n\n\n```python\n# Display expected counts; report chi-square, df, P, and the decision.\ndecision = \"Fail to Reject H0\"\nif(commute_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nprint(\"Chi-square =\",commute_test.statistic,\n      \"\\ndf =\",((len(observed_table[\"Car\"])-1)*(len(observed_table.loc[\"Day\"])-1)),\n      \"\\nP-value =\",commute_test.pvalue,\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision)\n\nprint(\"Expected count for public transit during the day =\",expected_count_ij,\"\\tEqual to respective value? -->\",expected_count_ij == expected_count_table[\"Public transit\"].loc[\"Day\"])\nexpected_count_table.head()\n```"},
    {"i": 54, "who": "tutor", "text": "Your crosstab, the hand-computed expected count, the stored `commute_test`, and the labeled expected-count table are all in place, and the KeyErrors are gone. [trimmed] (round chi-square to 3 and P to 4 decimals; percentage table *100 and row-sum check; smallest expected count check; np.isclose instead of ==; df cross-check; alpha caution.)"},
    {"i": 55, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_count_ij = (sum(observed_table.loc[\"Day\"])*(observed_table[\"Public transit\"].sum()))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\ncommute_test = stats.chi2_contingency(observed_table,correction=False)\nexpected_count_table = pd.DataFrame([\n    commute_test.expected_freq[:,0],\n    commute_test.expected_freq[:,1],\n    commute_test.expected_freq[:,2],\n],index=observed_table.index,\ncolumns=observed_table.columns)\n\nobserved_table.head()\n\n```\n\n\n```python\n(percentage_table*100).head()\n```\n\n\n```python\n# Display expected counts; report chi-square, df, P, and the decision.\ndecision = \"Fail to Reject H0\"\nif(commute_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nwarning_status = \"statistified\"\nif(min(commute_test.expected_freq)<5):\n    warning_status = \"VIOLATED\"\n\nprint(\"Chi-square =\",round(commute_test.statistic,3),\n      \"\\ndf =\",((len(observed_table[\"Car\"])-1)*(len(observed_table.loc[\"Day\"])-1)),\n      \"\\nP-value =\",round(commute_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n\nprint(\"Expected count for public transit during the day =\",expected_count_ij,\"\\tEqual to respective value? -->\",expected_count_ij == expected_count_table[\"Public transit\"].loc[\"Day\"])\nexpected_count_table.head()\n```\n\n---------------------------------------------------------------------------\nValueError                                Traceback (most recent call last)\nCell In[134], line 7\n      3 if(commute_test.pvalue < alpha):\n      4     decision = \"Reject H0\"\n      5 \n      6 warning_status = \"statistified\"\n----> 7 if(min(commute_test.expected_freq)<5):\n      8     warning_status = \"VIOLATED\"\n      9 \n     10 print(\"Chi-square =\",round(commute_test.statistic,3),\n\nValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()"},
    {"i": 56, "who": "tutor", "text": "That `ValueError` is the \"truth value of a Series/array is ambiguous\" error, and it's coming from one specific call: `min(commute_test.expected_freq)`. [trimmed] (built-in min compares rows; .min() on the array; np.isclose vs ==; row-sum check; label spelling; the new expected_count_table list of arrays makes columns into rows, so check layout.)"},
    {"i": 57, "who": "student", "code": true, "text": "```python\n# Display expected counts; report chi-square, df, P, and the decision.\ndecision = \"Fail to Reject H0\"\nif(commute_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nwarning_status = \"statistified\"\nif((commute_test.expected_freq).min()<5):\n    warning_status = \"VIOLATED\"\n\nprint(\"Chi-square =\",round(commute_test.statistic,3),\n      \"\\ndf =\",((len(observed_table[\"Car\"])-1)*(len(observed_table.loc[\"Day\"])-1)),\n      \"\\nP-value =\",round(commute_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n\nprint(\"Expected count for public transit during the day =\",expected_count_ij,\"\\tEqual to respective value? -->\",np.isclose(expected_count_ij, expected_count_table[\"Public transit\"].loc[\"Day\"]))\nexpected_count_table.head()\n```"},
    {"i": 58, "who": "tutor", "text": "The `.min()` fix works: `commute_test.expected_freq.min()` reduces the whole grid to one number. The `np.isclose` comparison is also what that check needs. [trimmed] (expected_count_table layout is transposed; fix options; row-sum check; label spelling; df cross-check.)"},
    {"i": 59, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_count_ij = (sum(observed_table.loc[\"Day\"])*(observed_table[\"Public transit\"].sum()))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\ncommute_test = stats.chi2_contingency(observed_table,correction=False)\nexpected_count_table = pd.DataFrame([\n    commute_test.expected_freq[0,:],\n    commute_test.expected_freq[1,:],\n    commute_test.expected_freq[2,:],\n],index=observed_table.index,\ncolumns=observed_table.columns)\n\nobserved_table.head()\n```"},
    {"i": 60, "who": "tutor", "text": "That fixes the layout. Each array in your list is now a *row* of `expected_freq`, so the Day row of `expected_count_table` lines up with the Day row of `observed_table`. [trimmed] (row-sum check and label spelling still open; questions to think through for the 3-4 sentences.)"},
    {"i": 61, "who": "student", "text": "```markdown\n$H_0$: time of day and mode of transportation and independent. $H_A$: the time of day and mode of transportation distributions differ. This dataset shows a decreasing number of commuters opting for public transit / walk or bike as the time of day goes from day --> evening --> night. Given that the p-value is less than $\\alpha = 0.05$, we reject $H_0$--we have statistically significant evidence that the time of day and mode of transportation distributions differ from one another. This study does not establish that changing an empoyee's shift causes a change in commute time, as the study did not randomly assign employees to shifts, preventing it from controlling for cofounding variables, such as commute preference or distance.\n```"},
    {"i": 62, "who": "tutor", "text": "Your draft covers all four things the prompt asks for: the hypotheses, the pattern, the test interpretation, and the causation question. [trimmed] (hypothesis wording and population; compare percentages rather than counts and mention the third mode; what a single chi-square does and does not say; random sampling vs assignment; typos; length.)"},
    {"i": 63, "who": "student", "text": "```markdown\n$H_0$: time of day and mode of transportation are independent. $H_A$: the time of day and mode of transportation distributions differ. This dataset shows a decreasing percentage of commuters opting for public transit / walk or bike as the time of day goes from day --> evening --> night. Given that the p-value is less than $\\alpha = 0.05$, we reject $H_0$--we have statistically significant evidence that the time of day and mode of transportation distributions differ from one another. This study does not establish that changing an empoyee's shift causes a change in commute time, as the study did not randomly assign employees to shifts, preventing it from controlling for cofounding variables, such as commute preference or distance.\n```"},
    {"i": 64, "who": "tutor", "text": "Switching from \"number\" to \"percentage\" fixes the main problem I raised. [trimmed] (list of remaining open items; offered a different-example explanation.)"},
    {"i": 65, "who": "student", "text": "```markdown\n**Answer:** $H_0$: shift and mode of transportation are independent. $H_A$: the shift and mode of transportation distributions differ. This dataset shows a decreasing percentage of commuters opting for public transit / walk or bike as the shift goes from day --> evening --> night. Given that the p-value is less than $\\alpha = 0.05$, we reject $H_0$--we have statistically significant evidence that the shift and mode of transportation distributions differ from one another. This study does not establish that changing an empoyee's shift causes a change in commute time, as the study did not randomly assign employees to shifts, preventing it from controlling for cofounding variables, such as commute preference or distance.\n```"},
    {"i": 66, "who": "tutor", "text": "\"Shift\" is in now, which closes one item. [trimmed] (remaining open items; code items still open; asked whether to revise, move to 3B, or make record.)"},
    {"i": 67, "who": "student", "text": "Can I use .append to add to an existing dataframe"},
    {"i": 68, "who": "tutor", "text": "No, not in current pandas. `DataFrame.append` was removed in pandas 2.0. [trimmed] (pd.concat and .loc replacements; lists for one-row DataFrames; matching column names; ignore_index; reassigning.)"},
    {"i": 69, "who": "student", "code": true, "text": "```python\nother_tests = pd.DataFrame({\n    \"factor\": [\n        \"Parking access\", \"Household vehicles\", \"Commute distance\",\n        \"Flexible start time\", \"Tenure\", \"Department\", \"Job category\",#\"Shift\",\n    ],\n    \"p_value\": [0.0012, 0.0041, 0.0170, 0.0410, 0.1800, 0.4600, 0.7300],#,commute_test.pvalue],\n})\n\n# Your code here\n\n# Add the shift result from part A to form a table of all eight tests.\npd.concat([other_tests,pd.DataFrame({\"factor\":\"Shift\",\"p-value\":commute_test.pvalue})],ignore_index=True)\n\n# Calculate the Bonferroni cutoff and add both significance indicators.\n# Use unrounded P-values for comparisons; round only for display.\n\nother_tests.head()\n```\n\n---------------------------------------------------------------------------\nValueError                                Traceback (most recent call last)\nCell In[174], line 12\n      8 \n      9 # Your code here\n     10 \n     11 # Add the shift result from part A to form a table of all eight tests.\n---> 12 pd.concat([other_tests,pd.DataFrame({\"factor\":\"Shift\",\"p-value\":commute_test.pvalue})],ignore_index=True)\n\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\frame.py:769, in DataFrame.__init__(self, data, index, columns, dtype, copy)\n...\n--> [634](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/internals/construction.py:634)     raise ValueError(\"If using all scalar values, you must pass an index\")\n\nValueError: If using all scalar values, you must pass an index"},
    {"i": 70, "who": "tutor", "text": "The error message is the one I described last time: `ValueError: If using all scalar values, you must pass an index`. [trimmed] (wrap scalars in lists; p-value vs p_value column name mismatch; concat result never assigned, so other_tests.head() still shows seven rows; use a new variable name.)"},
    {"i": 71, "who": "student", "text": "make my record"}
  ],
  "events": [
    {"i": 1, "part": "11-1A", "type": "explain"},
    {"i": 3, "part": "11-1A", "type": "explain"},
    {"i": 5, "part": "11-1A", "type": "explain"},
    {"i": 9, "part": "11-1A", "type": "code"},
    {"i": 11, "part": "11-1A", "type": "code"},
    {"i": 13, "part": "11-1A", "type": "code"},
    {"i": 15, "part": "11-1A", "type": "code"},
    {"i": 17, "part": "11-1A", "type": "code"},
    {"i": 21, "part": "11-1A", "type": "code"},
    {"i": 23, "part": "11-1A", "type": "explain"},
    {"i": 25, "part": "11-1A", "type": "code"},
    {"i": 27, "part": "11-1A", "type": "code"},
    {"i": 27, "part": "11-1A", "type": "part_pass"},
    {"i": 37, "part": "11-2A", "type": "code"},
    {"i": 39, "part": "11-2A", "type": "code"},
    {"i": 39, "part": "11-2A", "type": "part_pass"},
    {"i": 49, "part": "11-3A", "type": "code"},
    {"i": 49, "part": "11-3A", "type": "explain"},
    {"i": 51, "part": "11-3A", "type": "code"},
    {"i": 51, "part": "11-3A", "type": "explain"},
    {"i": 53, "part": "11-3A", "type": "code"},
    {"i": 55, "part": "11-3A", "type": "code"},
    {"i": 55, "part": "11-3A", "type": "explain"},
    {"i": 57, "part": "11-3A", "type": "code"},
    {"i": 59, "part": "11-3A", "type": "code"},
    {"i": 67, "part": "11-3B", "type": "explain"},
    {"i": 69, "part": "11-3B", "type": "code"},
    {"i": 69, "part": "11-3B", "type": "explain"}
  ],
  "parts": [
    {"part": "11-1A", "exchanges": 11, "max_tier": 3, "passed": true},
    {"part": "11-1B", "exchanges": 4, "max_tier": 3, "passed": false},
    {"part": "11-2A", "exchanges": 2, "max_tier": 2, "passed": true},
    {"part": "11-2B", "exchanges": 4, "max_tier": 3, "passed": false},
    {"part": "11-3A", "exchanges": 9, "max_tier": 3, "passed": false},
    {"part": "11-3B", "exchanges": 1, "max_tier": 2, "passed": false}
  ],
  "tutor_assessment": [
    "The student stalled longest on 11-1A's Survey 2 section (about eleven exchanges), mainly turning the weighted terms into a single weighted estimate, displaying percentages with area labels, and a Series-versus-DataFrame mix-up with set_index. In 11-3A the obstacles were confusing crosstab row labels with columns (two KeyErrors), a transposed expected-count table, and the min() on a 2-D array ValueError. The written answers also took several rounds, especially the within-area nonresponse assumption in 11-1B, the independence and repeated-test framing in 11-2B, and the workforce-level hypotheses and what a single chi-square says in 11-3A, which were still being revised when the session ended. In 11-3B, the student had just hit a scalar-DataFrame ValueError and a p-value column-name mismatch.",
    "No unstuck or release-valve use occurred in this session. The student's questions about tuples, arrays, pd.concat, and DataFrame.append were answered as free concept explanations, and the 'done' on 11-1A was accepted as a clear step rather than an escape-hatch use."
  ]
}
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA11",
  "export_n": 2,
  "session_date": "2026-11-16",
  "messages": [
    {"i": 1, "who": "student", "text": "How can I round the numbers displayed in a tuple?"},
    {"i": 2, "who": "tutor", "text": "Good question, and it comes up a lot with SciPy results. A tuple holds its values at full precision, and `print` shows all of them, so you format at the moment of display. [trimmed] (round() on each element, f-strings, formatting every element; round() changes stored value, f-strings do not; tuples are immutable; unpacking; pd.set_option only affects pandas display.)"},
    {"i": 3, "who": "student", "text": "Would it be easier to round a numpy array?"},
    {"i": 4, "who": "tutor", "text": "Often, yes. NumPy has a built-in for it, so you do not need a loop or a generator expression the way you do with a tuple. [trimmed] (np.round / .round, one setting for the whole array, np.set_printoptions, f-string formatting, pandas .round with a dict.)"},
    {"i": 5, "who": "student", "text": "How can I rename new columns during pd.concat"},
    {"i": 6, "who": "tutor", "text": "`pd.concat` does not rename columns itself. It matches columns **by name**, so what you rename is the thing you hand to it. [trimmed] (rename before/inside/after concat, mismatched names give NaN, ignore_index, keys=, axis=1, add_suffix, DataFrame.append removed in pandas 2.0.)"},
    {"i": 7, "who": "student", "text": "Did I miss anything for 1A?"},
    {"i": 8, "who": "tutor", "text": "I can check it, but I can't see your 1A cell yet. Could you paste the cell you've been working on, along with what it prints? [trimmed] (checklist of 1A prompt items.)"},
    {"i": 9, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"Margin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rates_unweighted = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_unweighted*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_unweighted,support_rates_weighted],axis=1,keys=[\"unweighted\",\"weighted\"])\ndisplay_table.head()\n\n```"},
    {"i": 10, "who": "tutor", "text": "Your Survey 1 logic is sound. [trimmed] (p̂ and SE not printed; group rates mislabeled unweighted; weighted terms never summed; Survey 2 not shown as percentages; row labels 0 and 1.)"},
    {"i": 11, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"Margin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_group,support_rates_weighted],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",support_rate_unweighted)\ndisplay_table.head()\n\n```"},
    {"i": 12, "who": "tutor", "text": "You fixed the unweighted estimate: `sum(responses[\"support\"]) / sum(responses[\"respondents\"])` is the right construction. [trimmed] (p̂ and SE still not printed; weighted still two terms; percent display; row labels.)"},
    {"i": 13, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1)\n    \"\\nMargin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_group,support_rates_weighted],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",support_rate_unweighted)\ndisplay_table.head()\n\n```"},
    {"i": 14, "who": "tutor", "text": "Nice progress: p̂ and the estimated SE now have their own printed lines. **This version will not run.** [trimmed] (missing comma after round(se_estimated*100,1) causes SyntaxError; margin label, weighted estimate, percent display, row labels still open.)"},
    {"i": 15, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1),\n    \"\\nMargin of Error [%] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\n\ndisplay_table = pd.concat([support_rates_group*100,support_rates_weighted*100],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2))\ndisplay_table.head()\n\n```"},
    {"i": 16, "who": "tutor", "text": "The missing comma is fixed, so the cell should run now, and you've started converting Survey 2 to percentages. [trimmed] (margin label, weighted estimate still two terms, row labels, unweighted display label.)"},
    {"i": 17, "who": "student", "code": true, "text": "How do you want percentage points to be done for the margin  of error? The units are already denotated as percentage and not proportion.\n\n```python\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\n#support_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rate_unweighted*responses[\"population_share\"]\n\n#display_table = pd.concat([support_rates_group*100,support_rates_weighted*100],axis=1,keys=[\"group\",\"weighted\"])\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2))\n#display_table.head()\n```"},
    {"i": 18, "who": "tutor", "text": "Your number is already right: `round(margin_of_error*100, 1)` gives the margin on the percentage scale, and that is exactly what \"percentage points\" means. The only change is the label. [trimmed] (percent vs percentage points; the weighted line now multiplies the overall unweighted rate by the shares instead of the group rates; generic .sum() example.)"},
    {"i": 19, "who": "student", "text": "What, specifically, was the issue with my response to survey 2?"},
    {"i": 20, "who": "tutor", "text": "There are three separate issues, and I'll name the exact lines. [trimmed] (wrong ingredient in weighted line; weighted estimate never reduced to one number; percentage display and row labels.)"},
    {"i": 21, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1),\n    \"\\nMargin of Error [% points] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\nsupport_rate_weighted = sum(support_rates_weighted)\n\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2),\n      \"\\nWighted proportion:\",round(support_rate_weighted*100,2))\n\n\n```"},
    {"i": 22, "who": "tutor", "text": "Both missing pieces are in now: the weighted estimate is a single number, `sum(support_rates_weighted)`, and the margin is labeled in percentage points. [trimmed] (group proportions not displayed; \"Wighted\" typo; naming note.)"},
    {"i": 23, "who": "student", "code": true, "text": "```python\nprint(support_rates_group.set_index(responses[\"area\"]))\n```\n\n---------------------------------------------------------------------------\nAttributeError                            Traceback (most recent call last)\nCell In[12], line 41\n     37 support_rate_weighted = sum(support_rates_weighted)\n     38 \n     39 print(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2),\n     40       \"\\nWighted proportion:\",round(support_rate_weighted*100,2))\n---> 41 print(support_rates_group.set_index(responses[\"area\"]))\n\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\generic.py:6194, in NDFrame.__getattr__(self, name)\n   6190             and name not in self._accessors\n   6191             and self._info_axis._can_hold_identifiers_and_holds_name(name)\n   6192         ):\n   6193             return self[name]\n-> [6194](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/generic.py:6194)         return object.__getattribute__(self, name)\n\nAttributeError: 'Series' object has no attribute 'set_index'"},
    {"i": 24, "who": "tutor", "text": "The error message is telling you exactly what's wrong: `support_rates_group` is a **Series** (one column of numbers), and `.set_index` is a **DataFrame** method. [trimmed] (three options for labeling a Series, alignment caution, percent formatting.)"},
    {"i": 25, "who": "student", "code": true, "text": "```python\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\npd.set_option(\"display.precision\", 4)\nalpha = 0.05\n\n# Survey 1: complete responses to a simple random sample.\nn = 900\nsupport = 513\n\n# Survey 2: separate responses; population shares are supplied.\nresponses = pd.DataFrame({\n    \"area\": [\"Within 3 km\", \"Farther away\"],\n    \"population_share\": [0.40, 0.60],\n    \"respondents\": [600, 300],\n    \"support\": [420, 120],\n})\ndisplay(responses)\n\n# Your code here\n# Survey 1: calculate p_hat, the estimated SE, margin, and interval.\np_hat = support/n\nse_estimated = (p_hat*(1.0-p_hat)/n)**(1/2)\nmargin_of_error = 2*se_estimated\ninterval = np.array([p_hat-margin_of_error,p_hat+margin_of_error])\n\nprint(\"p_hat [%] =\",round(p_hat*100,1),\n    \"\\nEstimated SE [%] =\",round(se_estimated*100,1),\n    \"\\nMargin of Error [% points] =\",round(margin_of_error*100,1),\n    \"\\n95% confidence interval for percentage of adults supporting the library hours proposal:\",(interval*100).round(1))\n\n# Survey 2: calculate group support rates and unweighted/weighted estimates.\nsupport_rate_unweighted = sum(responses[\"support\"])/sum(responses[\"respondents\"])\n\nsupport_rates_group = responses[\"support\"]/responses[\"respondents\"]\nsupport_rates_weighted = support_rates_group*responses[\"population_share\"]\nsupport_rate_weighted = sum(support_rates_weighted)\n\nsupport_rates_group.index = responses[\"area\"]\nprint(\"\\nUnweighted proportion:\",round(support_rate_unweighted*100,2),\n      \"\\nWighted proportion:\",round(support_rate_weighted*100,2))\nprint(\"\\nGroup proportions:\\n\",support_rates_group)\n\n```"},
    {"i": 26, "who": "tutor", "text": "Your group proportions now carry the area names. [trimmed] (order-of-assignment alignment caution; group proportions still not percentages; typo; overall-line labels.)"},
    {"i": 27, "who": "student", "code": true, "text": "```python\nsupport_rates_group.index = responses[\"area\"]\nprint(\"\\nUnweighted proportion [%]:\",round(support_rate_unweighted*100,2),\n      \"\\nWighted proportion [%]:\",round(support_rate_weighted*100,2))\nprint(\"\\nGroup proportion [%]:\\n\",support_rates_group*100)\n```\n\ndone"},
    {"i": 28, "who": "tutor", "text": "Those lines cover what the prompt asks: the group proportions now print as percentages with area names, and both overall estimates carry the `[%]` label. [trimmed] (typo note; Restart and Run All; offered 1B or record.)"},
    {"i": 29, "who": "student", "text": "```markdown\n**Answer:**\n1. “Approximately 95% confidence” means that if we repeated our sampling procedure, we would expect 95% of confidence intervals constructed in such a manner to contain the true population proportion of people who support the library hour proposal.\n2. The direction of change from weighted to unweighted was negative--i.e. the proportion estimate decreased. The \"Within 3 km\" group was overrepresented among the respondents because they had a larger share of respondents despite having a smaller population share.\n3. Assuming that nonresponse rates are equal for both areas is neccesary in order for weighting to remove the nonresponse bias. The ordinary random-sampling margin of error would not be able to account for any violation of this assumption because it does not take into account the fact that the results are biased in its formula. \n```"},
    {"i": 30, "who": "tutor", "text": "Thanks for posting a draft. Per the prompt, your answer needs to be graded as your own writing, so I'll point at where the reasoning holds and where it needs another look, without rewriting anything. [trimmed] (Q1 solid; Q2 direction wording and reason; Q3 assumption is about people within one area, not equal response rates; length.)"},
    {"i": 31, "who": "student", "text": "```markdown\n2. The direction of change from unweighted to weighted was negative--i.e. the proportion estimate decreased. This change is direction is due to the \"Within 3 km\" group being overrepresented among the respondents (they had a larger share of respondents despite having a smaller population share).\n3. Assuming that the nonrespondents would have the rates of support for the proposal is neccesary in order for weighting to remove bias. The ordinary random-sampling margin of error would not be able to account for any violation of this assumption because it does not take into account the fact that the results are biased in its formula. Bias is a systematic error that can not be correctd by a margin of error.\n```"},
    {"i": 32, "who": "tutor", "text": "Your revision is closer, and the direction wording in question 2 now matches the prompt. [trimmed] (Q2 needs the link to the group support rates; Q3 sentence incomplete; margin-of-error reasoning strong; length.)"},
    {"i": 33, "who": "student", "text": "```markdown\n3. Assuming that the nonrespondents would have the same rates of support for the proposal is neccesary in order for weighting to remove bias.\n```"},
    {"i": 34, "who": "tutor", "text": "You've got the core of it. The assumption is about equal support rates, and you now say it is what weighting needs. [trimmed] (name the comparison group and that it holds within each area; typo; assemble all three answers.)"},
    {"i": 35, "who": "student", "text": "```markdown\nAssuming that the nonrespondents would have the same rates of support for the proposal as those who responded is neccesary in order for weighting to remove bias.\n```"},
    {"i": 36, "who": "tutor", "text": "That is the missing piece: you now name the comparison, nonrespondents against respondents. [trimmed] (within each area still not stated; typo; assemble; LE11-2 next.)"},
    {"i": 37, "who": "student", "code": true, "text": "```python\nservers = [\"A\", \"B\", \"C\", \"D\"]\nprobabilities = np.array([0.40, 0.30, 0.20, 0.10])\nobserved = np.array([183, 112, 79, 26])\n\n# Your code here\n# Calculate expected counts using observed.sum() and probabilities.\nexpected = probabilities*observed.sum()\n\n# Build the contributions table and calculate chi-square manually.\ncontributions = ((observed-expected)**2)/expected\nchi_statistic = sum(contributions)\nprint(\"Chi-square statistic (manual) =\",round(chi_statistic,3))\n\nchi_table = pd.DataFrame({\n    \"Server\":servers,\n    \"Observed counts\":observed,\n    \"Expected counts\":expected,\n    \"Chi-square contribution\":contributions,\n})\n\n# Run stats.chisquare(...), report df and P, and check expected counts.\nalpha = 0.05\nchi_test = stats.chisquare(f_obs=observed,f_exp=expected)\n\ndecision = \"Fail to Reject H0\"\nif(chi_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nwarning_status = \"statistified\"\nfor item in expected:\n    if(item <= 5):\n        warning_status = \"VIOLATED\"\n\nprint(\"Chi-square statistic (stats.chisquare(...)) =\",round(chi_test.statistic,3),\n      \"\\ndf =\",len(probabilities)-1,\n      \"\\nP-value =\",round(chi_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n\n\nchi_table.head()\n```"},
    {"i": 38, "who": "tutor", "text": "Your LE11-2A cell covers each item the prompt lists, and the logic is sound. [trimmed] (rule-of-thumb should be < 5 not <= 5; explicit match check with np.isclose; typo; keep alpha unchanged later.)"},
    {"i": 39, "who": "student", "code": true, "text": "```python\nwarning_status = \"statistified\"\nfor item in expected:\n    if(item < 5):\n        warning_status = \"VIOLATED\"\n\nequality_status = np.isclose(chi_statistic, chi_test.statistic)\n\nprint(\"Chi-square statistic (stats.chisquare(...)) =\",round(chi_test.statistic,3), \"\\tMatches manual? -->\",equality_status,\n      \"\\ndf =\",len(probabilities)-1,\n      \"\\nP-value =\",round(chi_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n```"},
    {"i": 40, "who": "tutor", "text": "Both changes are in. `item < 5` now matches \"at least 5\", and `np.isclose(...)` compares the unrounded values. That covers every item on the LE11-2A list. [trimmed] (label typo; Restart and Run All; listed the three LE11-2B questions.)"},
    {"i": 41, "who": "student", "text": "```markdown\n**Answer:**\n1. $H_0$: The router assigns packets to routers A, B, C, and D according to their expected probabilities of 0.40, 0.30, 0.20, and 0.10, respectively. $H_A$: The router does not assign packets to the routers according to said expected probabilities.\n2. Server D contributs the most to $\\chi^2$. It is underrepresented relative to its expected count.\n3. The p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme as the one obtained in the sample under $H_0$. It does not provide the probability that the router's configured probabilities are correct.\n```"},
    {"i": 42, "who": "tutor", "text": "Your draft is in good shape, and the core ideas are right. [trimmed] (Q1 missing the 5% conclusion, servers not routers; Q2 consistent with reference; Q3 missing independence and tail check; length; typo.)"},
    {"i": 43, "who": "student", "text": "```markdown\n1. $H_0$: The router assigns packets to routers A, B, C, and D according to their expected probabilities of 0.40, 0.30, 0.20, and 0.10, respectively. $H_A$: The router does not assign packets to the routers according to said expected probabilities. Conclusion at $\\alpha = 0.05$: since $P < \\alpha$, we reject $H_0$. We have statistically significant evidence that the router does not assign packets to the routers according to said expected probabilities.\n2. Server D contributes the most to $\\chi^2$. It is underrepresented relative to its expected count.\n3. The p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme (greater) as the one obtained in the sample under $H_0$ (meaning the servers are expected to truly asssign packets according to the expected probabilities). It does not provide the probability that the router's configured probabilities are correct.\n```"},
    {"i": 44, "who": "tutor", "text": "Question 1 is now complete: the hypotheses are about the assignment probabilities, and your conclusion matches your own P and α. [trimmed] (servers wording; Q3 independence and repeated-test framing; typos; length.)"},
    {"i": 45, "who": "student", "text": "```markdown\n3. The p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme (greater) as the one obtained in the sample under $H_0$ (meaning the router is expected to truly asssign packets according to the expected probabilities). It does not provide the probability that the router's configured probabilities are correct.\n```"},
    {"i": 46, "who": "tutor", "text": "You're close on question 3. [trimmed] (independence still missing; what gets repeated; typos; offered a different-example walkthrough.)"},
    {"i": 47, "who": "student", "text": "```markdown\nThe p-value in this case represents the chance of obtaining a chi-square statistic as or more extreme (greater) as the one obtained in the sample of independent packets under $H_0$ (meaning the router is expected to truly asssign packets according to the expected probabilities).\n```"},
    {"i": 48, "who": "tutor", "text": "Independence is in now, and your sentence says the packets are independent and that the calculation is done under H₀. [trimmed] (dropped the not-the-probability-configured-is-correct sentence; typos; asked about LE11-3 or record.)"},
    {"i": 49, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.concat([observed_table.loc[observed_table[\"shift\"]==\"Day\"]/sum(observed_table.loc[observed_table[\"shift\"]==\"Day\"]),\n                              observed_table.loc[observed_table[\"shift\"]==\"Evening\"]/sum(observed_table.loc[observed_table[\"shift\"]==\"Evening\"]),\n                              observed_table.loc[observed_table[\"shift\"]==\"Night\"]/sum(observed_table.loc[observed_table[\"shift\"]==\"Night\"]),])\n\n# Calculate the Day x Public transit expected count from the margins.\n# Run the chi-square test and keep the result in commute_test.\n# Display expected counts; report chi-square, df, P, and the decision.\n\nobserved_table.head()\n\n```\n\n---------------------------------------------------------------------------\nKeyError                                  Traceback (most recent call last)\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\indexes\\base.py:3641, in Index.get_loc(self, key)\n   3640 try:\n-> [3641](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/indexes/base.py:3641)     return self._engine.get_loc(casted_key)\n   3642 except KeyError as err:\n\nKeyError: 'shift'\n\nThe above exception was the direct cause of the following exception:\n\nKeyError                                  Traceback (most recent call last)\nCell In[52], line 7\n...\nKeyError: 'shift'\nOutput is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings..."},
    {"i": 50, "who": "tutor", "text": "Your `pd.crosstab(commutes[\"shift\"], commutes[\"commute_mode\"])` line is right, and the bracket access to `shift` is exactly what it needs. The error comes from the next statement. [trimmed] (after crosstab, shift is the row index not a column; .loc[\"Day\"]; normalize=\"index\"; percentages need *100.)"},
    {"i": 51, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_total_ij = (sum(observed_table.loc[\"Day\"])*(sum(observed_table.loc[\"Car\"])))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\n\n\n# Display expected counts; report chi-square, df, P, and the decision.\n\n\n\nobserved_table.head()\n```\n\n---------------------------------------------------------------------------\nKeyError                                  Traceback (most recent call last)\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\indexes\\base.py:3641, in Index.get_loc(self, key)\n   3640 try:\n-> [3641](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/indexes/base.py:3641)     return self._engine.get_loc(casted_key)\n   3642 except KeyError as err:\n\nKeyError: 'Car'\n\nThe above exception was the direct cause of the following exception:\n\nKeyError                                  Traceback (most recent call last)\nCell In[79], line 10\n...\nKeyError: 'Car'"},
    {"i": 52, "who": "tutor", "text": "Your crosstab lines are right now. `normalize = \"index\"` is the setting that divides each row by its own row total. The error is on the expected-count line. [trimmed] (.loc takes row labels; columns use brackets; wrong column Car vs Public transit; N alternatives; percentages need *100 and row-sum check.)"},
    {"i": 53, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_count_ij = (sum(observed_table.loc[\"Day\"])*(observed_table[\"Public transit\"].sum()))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\ncommute_test = stats.chi2_contingency(observed_table,correction=False)\nexpected_count_table = pd.DataFrame({\n    \"Car\":commute_test.expected_freq[:,0],\n    \"Public transit\":commute_test.expected_freq[:,1],\n    \"Walk or bike\":commute_test.expected_freq[:,2],\n},index=[\"Day\",\"Evening\",\"Night\"])\n\nobserved_table.head()\n```\n\n\n```python\npercentage_table.head()\n```\n\n\n```python\n# Display expected counts; report chi-square, df, P, and the decision.\ndecision = \"Fail to Reject H0\"\nif(commute_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nprint(\"Chi-square =\",commute_test.statistic,\n      \"\\ndf =\",((len(observed_table[\"Car\"])-1)*(len(observed_table.loc[\"Day\"])-1)),\n      \"\\nP-value =\",commute_test.pvalue,\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision)\n\nprint(\"Expected count for public transit during the day =\",expected_count_ij,\"\\tEqual to respective value? -->\",expected_count_ij == expected_count_table[\"Public transit\"].loc[\"Day\"])\nexpected_count_table.head()\n```"},
    {"i": 54, "who": "tutor", "text": "Your crosstab, the hand-computed expected count, the stored `commute_test`, and the labeled expected-count table are all in place, and the KeyErrors are gone. [trimmed] (round chi-square to 3 and P to 4 decimals; percentage table *100 and row-sum check; smallest expected count check; np.isclose instead of ==; df cross-check; alpha caution.)"},
    {"i": 55, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_count_ij = (sum(observed_table.loc[\"Day\"])*(observed_table[\"Public transit\"].sum()))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\ncommute_test = stats.chi2_contingency(observed_table,correction=False)\nexpected_count_table = pd.DataFrame([\n    commute_test.expected_freq[:,0],\n    commute_test.expected_freq[:,1],\n    commute_test.expected_freq[:,2],\n],index=observed_table.index,\ncolumns=observed_table.columns)\n\nobserved_table.head()\n\n```\n\n\n```python\n(percentage_table*100).head()\n```\n\n\n```python\n# Display expected counts; report chi-square, df, P, and the decision.\ndecision = \"Fail to Reject H0\"\nif(commute_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nwarning_status = \"statistified\"\nif(min(commute_test.expected_freq)<5):\n    warning_status = \"VIOLATED\"\n\nprint(\"Chi-square =\",round(commute_test.statistic,3),\n      \"\\ndf =\",((len(observed_table[\"Car\"])-1)*(len(observed_table.loc[\"Day\"])-1)),\n      \"\\nP-value =\",round(commute_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n\nprint(\"Expected count for public transit during the day =\",expected_count_ij,\"\\tEqual to respective value? -->\",expected_count_ij == expected_count_table[\"Public transit\"].loc[\"Day\"])\nexpected_count_table.head()\n```\n\n---------------------------------------------------------------------------\nValueError                                Traceback (most recent call last)\nCell In[134], line 7\n      3 if(commute_test.pvalue < alpha):\n      4     decision = \"Reject H0\"\n      5 \n      6 warning_status = \"statistified\"\n----> 7 if(min(commute_test.expected_freq)<5):\n      8     warning_status = \"VIOLATED\"\n      9 \n     10 print(\"Chi-square =\",round(commute_test.statistic,3),\n\nValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()"},
    {"i": 56, "who": "tutor", "text": "That `ValueError` is the \"truth value of a Series/array is ambiguous\" error, and it's coming from one specific call: `min(commute_test.expected_freq)`. [trimmed] (built-in min compares rows; .min() on the array; np.isclose vs ==; row-sum check; label spelling; the new expected_count_table list of arrays makes columns into rows, so check layout.)"},
    {"i": 57, "who": "student", "code": true, "text": "```python\n# Display expected counts; report chi-square, df, P, and the decision.\ndecision = \"Fail to Reject H0\"\nif(commute_test.pvalue < alpha):\n    decision = \"Reject H0\"\n\nwarning_status = \"statistified\"\nif((commute_test.expected_freq).min()<5):\n    warning_status = \"VIOLATED\"\n\nprint(\"Chi-square =\",round(commute_test.statistic,3),\n      \"\\ndf =\",((len(observed_table[\"Car\"])-1)*(len(observed_table.loc[\"Day\"])-1)),\n      \"\\nP-value =\",round(commute_test.pvalue,4),\n      \"\\nDecision at alpha =\",alpha,\"-->\",decision,\n      \"\\nAll expected counts >= 5? -->\",warning_status)\n\nprint(\"Expected count for public transit during the day =\",expected_count_ij,\"\\tEqual to respective value? -->\",np.isclose(expected_count_ij, expected_count_table[\"Public transit\"].loc[\"Day\"]))\nexpected_count_table.head()\n```"},
    {"i": 58, "who": "tutor", "text": "The `.min()` fix works: `commute_test.expected_freq.min()` reduces the whole grid to one number. The `np.isclose` comparison is also what that check needs. [trimmed] (expected_count_table layout is transposed; fix options; row-sum check; label spelling; df cross-check.)"},
    {"i": 59, "who": "student", "code": true, "text": "```python\n# Your code here\n\n# Build observed_table using pd.crosstab(...).\nobserved_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"])\n\n# Calculate percentages within each shift (not percentages of all employees).\npercentage_table = pd.crosstab(commutes[\"shift\"],commutes[\"commute_mode\"],normalize = \"index\")\n\n# Calculate the Day x Public transit expected count from the margins.\nexpected_count_ij = (sum(observed_table.loc[\"Day\"])*(observed_table[\"Public transit\"].sum()))/(sum(observed_table.loc[\"Day\"]) + sum(observed_table.loc[\"Evening\"]) + sum(observed_table.loc[\"Night\"]))\n\n# Run the chi-square test and keep the result in commute_test.\ncommute_test = stats.chi2_contingency(observed_table,correction=False)\nexpected_count_table = pd.DataFrame([\n    commute_test.expected_freq[0,:],\n    commute_test.expected_freq[1,:],\n    commute_test.expected_freq[2,:],\n],index=observed_table.index,\ncolumns=observed_table.columns)\n\nobserved_table.head()\n```"},
    {"i": 60, "who": "tutor", "text": "That fixes the layout. Each array in your list is now a *row* of `expected_freq`, so the Day row of `expected_count_table` lines up with the Day row of `observed_table`. [trimmed] (row-sum check and label spelling still open; questions to think through for the 3-4 sentences.)"},
    {"i": 61, "who": "student", "text": "```markdown\n$H_0$: time of day and mode of transportation and independent. $H_A$: the time of day and mode of transportation distributions differ. This dataset shows a decreasing number of commuters opting for public transit / walk or bike as the time of day goes from day --> evening --> night. Given that the p-value is less than $\\alpha = 0.05$, we reject $H_0$--we have statistically significant evidence that the time of day and mode of transportation distributions differ from one another. This study does not establish that changing an empoyee's shift causes a change in commute time, as the study did not randomly assign employees to shifts, preventing it from controlling for cofounding variables, such as commute preference or distance.\n```"},
    {"i": 62, "who": "tutor", "text": "Your draft covers all four things the prompt asks for: the hypotheses, the pattern, the test interpretation, and the causation question. [trimmed] (hypothesis wording and population; compare percentages rather than counts and mention the third mode; what a single chi-square does and does not say; random sampling vs assignment; typos; length.)"},
    {"i": 63, "who": "student", "text": "```markdown\n$H_0$: time of day and mode of transportation are independent. $H_A$: the time of day and mode of transportation distributions differ. This dataset shows a decreasing percentage of commuters opting for public transit / walk or bike as the time of day goes from day --> evening --> night. Given that the p-value is less than $\\alpha = 0.05$, we reject $H_0$--we have statistically significant evidence that the time of day and mode of transportation distributions differ from one another. This study does not establish that changing an empoyee's shift causes a change in commute time, as the study did not randomly assign employees to shifts, preventing it from controlling for cofounding variables, such as commute preference or distance.\n```"},
    {"i": 64, "who": "tutor", "text": "Switching from \"number\" to \"percentage\" fixes the main problem I raised. [trimmed] (list of remaining open items; offered a different-example explanation.)"},
    {"i": 65, "who": "student", "text": "```markdown\n**Answer:** $H_0$: shift and mode of transportation are independent. $H_A$: the shift and mode of transportation distributions differ. This dataset shows a decreasing percentage of commuters opting for public transit / walk or bike as the shift goes from day --> evening --> night. Given that the p-value is less than $\\alpha = 0.05$, we reject $H_0$--we have statistically significant evidence that the shift and mode of transportation distributions differ from one another. This study does not establish that changing an empoyee's shift causes a change in commute time, as the study did not randomly assign employees to shifts, preventing it from controlling for cofounding variables, such as commute preference or distance.\n```"},
    {"i": 66, "who": "tutor", "text": "\"Shift\" is in now, which closes one item. [trimmed] (remaining open items; code items still open; asked whether to revise, move to 3B, or make record.)"},
    {"i": 67, "who": "student", "text": "Can I use .append to add to an existing dataframe"},
    {"i": 68, "who": "tutor", "text": "No, not in current pandas. `DataFrame.append` was removed in pandas 2.0. [trimmed] (pd.concat and .loc replacements; lists for one-row DataFrames; matching column names; ignore_index; reassigning.)"},
    {"i": 69, "who": "student", "code": true, "text": "```python\nother_tests = pd.DataFrame({\n    \"factor\": [\n        \"Parking access\", \"Household vehicles\", \"Commute distance\",\n        \"Flexible start time\", \"Tenure\", \"Department\", \"Job category\",#\"Shift\",\n    ],\n    \"p_value\": [0.0012, 0.0041, 0.0170, 0.0410, 0.1800, 0.4600, 0.7300],#,commute_test.pvalue],\n})\n\n# Your code here\n\n# Add the shift result from part A to form a table of all eight tests.\npd.concat([other_tests,pd.DataFrame({\"factor\":\"Shift\",\"p-value\":commute_test.pvalue})],ignore_index=True)\n\n# Calculate the Bonferroni cutoff and add both significance indicators.\n# Use unrounded P-values for comparisons; round only for display.\n\nother_tests.head()\n```\n\n---------------------------------------------------------------------------\nValueError                                Traceback (most recent call last)\nCell In[174], line 12\n      8 \n      9 # Your code here\n     10 \n     11 # Add the shift result from part A to form a table of all eight tests.\n---> 12 pd.concat([other_tests,pd.DataFrame({\"factor\":\"Shift\",\"p-value\":commute_test.pvalue})],ignore_index=True)\n\nFile ~\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pandas\\core\\frame.py:769, in DataFrame.__init__(self, data, index, columns, dtype, copy)\n...\n--> [634](https://file+.vscode-resource.vscode-cdn.net/c%3A/Users/julia/OneDrive%20-%20University%20of%20Pittsburgh/Classes/Semester%203/ENGR%201451/Lab%20Submissions/~/AppData/Local/Python/pythoncore-3.14-64/Lib/site-packages/pandas/core/internals/construction.py:634)     raise ValueError(\"If using all scalar values, you must pass an index\")\n\nValueError: If using all scalar values, you must pass an index"},
    {"i": 70, "who": "tutor", "text": "The error message is the one I described last time: `ValueError: If using all scalar values, you must pass an index`. [trimmed] (wrap scalars in lists; p-value vs p_value column name mismatch; concat result never assigned, so other_tests.head() still shows seven rows; use a new variable name.)"},
    {"i": 71, "who": "student", "text": "make my record"},
    {"i": 72, "who": "student", "code": true, "text": "```python\n# Calculate the Bonferroni cutoff and add both significance indicators.\nalpha_each = alpha/len(all_tests)\nmask1 = (all_tests[\"p_value\"] < alpha_each)\n\nall_tests_with_significance = pd.concat([all_tests,mask1],axis=1)\nall_tests_with_significance.sort_values(\"p_value\",ascending=True).head(9)\n```"},
    {"i": 73, "who": "tutor", "text": "The Bonferroni cutoff is built the right way: `alpha / len(all_tests)` ties the divisor to the table size, and a separate name like `alpha_each` leaves `alpha` alone. [trimmed] (mask1 keeps the name p_value so concat creates a duplicate column and sort_values would fail; assign a named column instead; only one of the two requested indicators present; .head(9) note; check all_tests has eight rows and unrounded shift P.)"},
    {"i": 74, "who": "student", "code": true, "text": "```python\n# Calculate the Bonferroni cutoff and add both significance indicators.\nalpha_each = alpha/len(all_tests)\nmask1 = (all_tests[\"p_value\"] < alpha_each)\n\nprint(alpha_each)\n\nall_tests_with_significance = pd.concat([all_tests,mask1],axis=1,columns=[all_tests.columns,\"Still Significant?\"])\nall_tests_with_significance.head(9)\n```\n\n---------------------------------------------------------------------------\nTypeError                                 Traceback (most recent call last)\nCell In[227], line 7\n      3 mask1 = (all_tests[\"p_value\"] < alpha_each)\n      4 \n      5 print(alpha_each)\n      6 \n----> 7 all_tests_with_significance = pd.concat([all_tests,mask1],axis=1,columns=[all_tests.columns,\"Still Significant?\"])\n      8 all_tests_with_significance.head(9)\n\nTypeError: concat() got an unexpected keyword argument 'columns'"},
    {"i": 75, "who": "tutor", "text": "The error says it directly: `pd.concat` has no `columns=` argument. [trimmed] (columns= belongs to pd.DataFrame; assign named boolean columns instead of concat; duplicate-name caution; second indicator still missing; confirm all_tests has eight rows.)"},
    {"i": 76, "who": "student", "code": true, "text": "```python\nother_tests = pd.DataFrame({\n    \"factor\": [\n        \"Parking access\", \"Household vehicles\", \"Commute distance\",\n        \"Flexible start time\", \"Tenure\", \"Department\", \"Job category\",\n    ],\n    \"p_value\": [0.0012, 0.0041, 0.0170, 0.0410, 0.1800, 0.4600, 0.7300],\n})\n\n# Use unrounded P-values for comparisons; round only for display.\n\n\n# Your code here\n\n# Add the shift result from part A to form a table of all eight tests.\nall_tests = pd.concat([other_tests,pd.DataFrame({\"factor\":[\"Shift\"],\"p_value\":[commute_test.pvalue]})],ignore_index=True)\n\nall_tests.head(9)\n```\n\n\n```python\n# Calculate the Bonferroni cutoff and add both significance indicators.\nalpha_each = alpha/len(all_tests)\nall_tests[\"Still significant and alpha_each?\"] = (all_tests[\"p_value\"] < alpha_each)\n\nprint(\"Boneferroni cutoff (alpha_each):\",alpha_each)\n\nall_tests.sort_values(\"p_value\").head(9)\n```\n\nAm I missing anything for 3B?"},
    {"i": 77, "who": "tutor", "text": "You're nearly there. The table is built correctly, and one required piece is missing. [trimmed] (all_tests built correctly with unrounded shift P; cutoff correct; missing the ordinary 0.05 indicator column; label readability; \"Boneferroni\" typo; keep alpha_each separate; 2-3 sentences are the student's; offered a false-alarm explanation.)"},
    {"i": 78, "who": "student", "code": true, "text": "```python\n# Calculate the Bonferroni cutoff and add both significance indicators.\nalpha_each = alpha/len(all_tests)\nall_tests[\"Intially significant at alpha = 0.05?\"] = (all_tests[\"p_value\"] < alpha)\nall_tests[\"Still significant and alpha_each?\"] = (all_tests[\"p_value\"] < alpha_each)\n\nprint(\"Boneferroni cutoff (alpha_each):\",alpha_each)\n\nall_tests.sort_values(\"p_value\").head(9)\n```"},
    {"i": 79, "who": "tutor", "text": "Both indicators are in now, so the table covers what the prompt asks for. [trimmed] (typos \"Intially\" and \"Boneferroni\"; second column label hard to read; confirm eight rows; questions to consider before writing the 2-3 sentences; offered a different-example explanation of testing many factors.)"},
    {"i": 80, "who": "student", "text": "```markdown\nThe conclusions that change from being significant at $\\alpha = 0.05$ to no longer being significant at $\\alpha_{\\mathrm{each}} = 0.00625$ are shift, commute distance, and flexible start time (parking access and houshold vehicles remain significant). The ordinary $P<0.05$ results would misrepresent the search because they dont account for the act that, under $H_0$, if we run a large amount of significane tests, we are likley to find a few p-values that are less than 0.05 (this is why the Boneferroni cutoff is neccesary).\nIf a result fails the adjusted cutoff, it does not prove there is no association, rather, it does not provide evidence in support of an association (there may still be a weak or miniscule correlation that the test was unable to pick up on).\n```"},
    {"i": 81, "who": "tutor", "text": "Your list of changed conclusions matches my reference calculation. [trimmed] (changed conclusions correct and shift named; misrepresentation answer explains why a cutoff is needed but not what a reader of a P<0.05-only report is missing; \"correlation\" vs association; what limits a test's ability to detect; typos; length fine; reminder of open 3A items.)"},
    {"i": 82, "who": "student", "text": "```markdown\nThe conclusions that change from being significant at $\\alpha = 0.05$ to no longer being significant at $\\alpha_{\\mathrm{each}} = 0.00625$ are shift, commute distance, and flexible start time (parking access and houshold vehicles remain significant). The ordinary $P<0.05$ results would misrepresent the search because it would hide from the reader the fact that, under $H_0$, if we run a large amount of significane tests, we are likley to find a few p-values that are less than 0.05 (this is why the Boneferroni cutoff is neccesary). If a result fails the adjusted cutoff, it does not prove there is no association, rather, it does not provide evidence in support of an association (there may still be a weak association that the test was unable to pick up on).\n```"},
    {"i": 83, "who": "tutor", "text": "This version fits the prompt much better, and you're at three sentences, within the 2-3 asked for. [trimmed] (reporting point still needs the number of tests a reader would not know about; \"large amount\" vs the family size; optional note on what limits detection; typos; open 3A items; Restart and Run All.)"},
    {"i": 84, "who": "student", "text": "Is anything anywhere in my notebook missing or flat out incorrect?"},
    {"i": 85, "who": "tutor", "text": "I can only judge what you've pasted in this chat, not the saved notebook. [trimmed] (no wrong numerical results found; open code items: 3A row-sum check, \"statistified\" label, typos, 3B column labels, file name/path agreement for the CSV; open written items for 1B, 2B, 3A, 3B; noted that the earlier record had the wrong session date of 2026-11-15 and should be replaced by a new record.)"},
    {"i": 86, "who": "student", "text": "make my record"}
  ],
  "events": [
    {"i": 1, "part": "11-1A", "type": "explain"},
    {"i": 3, "part": "11-1A", "type": "explain"},
    {"i": 5, "part": "11-1A", "type": "explain"},
    {"i": 9, "part": "11-1A", "type": "code"},
    {"i": 11, "part": "11-1A", "type": "code"},
    {"i": 13, "part": "11-1A", "type": "code"},
    {"i": 15, "part": "11-1A", "type": "code"},
    {"i": 17, "part": "11-1A", "type": "code"},
    {"i": 21, "part": "11-1A", "type": "code"},
    {"i": 23, "part": "11-1A", "type": "explain"},
    {"i": 25, "part": "11-1A", "type": "code"},
    {"i": 27, "part": "11-1A", "type": "code"},
    {"i": 27, "part": "11-1A", "type": "part_pass"},
    {"i": 37, "part": "11-2A", "type": "code"},
    {"i": 39, "part": "11-2A", "type": "code"},
    {"i": 39, "part": "11-2A", "type": "part_pass"},
    {"i": 49, "part": "11-3A", "type": "code"},
    {"i": 49, "part": "11-3A", "type": "explain"},
    {"i": 51, "part": "11-3A", "type": "code"},
    {"i": 51, "part": "11-3A", "type": "explain"},
    {"i": 53, "part": "11-3A", "type": "code"},
    {"i": 55, "part": "11-3A", "type": "code"},
    {"i": 55, "part": "11-3A", "type": "explain"},
    {"i": 57, "part": "11-3A", "type": "code"},
    {"i": 59, "part": "11-3A", "type": "code"},
    {"i": 67, "part": "11-3B", "type": "explain"},
    {"i": 69, "part": "11-3B", "type": "code"},
    {"i": 69, "part": "11-3B", "type": "explain"},
    {"i": 72, "part": "11-3B", "type": "code"},
    {"i": 74, "part": "11-3B", "type": "code"},
    {"i": 74, "part": "11-3B", "type": "explain"},
    {"i": 76, "part": "11-3B", "type": "code"},
    {"i": 78, "part": "11-3B", "type": "code"}
  ],
  "parts": [
    {"part": "11-1A", "exchanges": 11, "max_tier": 3, "passed": true},
    {"part": "11-1B", "exchanges": 4, "max_tier": 3, "passed": false},
    {"part": "11-2A", "exchanges": 2, "max_tier": 2, "passed": true},
    {"part": "11-2B", "exchanges": 4, "max_tier": 3, "passed": false},
    {"part": "11-3A", "exchanges": 9, "max_tier": 3, "passed": false},
    {"part": "11-3B", "exchanges": 7, "max_tier": 3, "passed": false}
  ],
  "tutor_assessment": [
    "The student stalled longest on 11-1A's Survey 2 section (about eleven exchanges): turning the weighted terms into one weighted estimate, displaying percentages with area labels, and a Series-versus-DataFrame set_index mix-up. In 11-3A the obstacles were confusing crosstab row labels with columns (two KeyErrors), a transposed expected-count table, and min() on a 2-D array. In 11-3B the obstacles were building a one-row DataFrame from scalars, a column-name mismatch, an unassigned concat result, and using concat with a Series and a nonexistent columns= argument; the table and both indicators were eventually correct. The written answers also took several rounds, especially the within-area nonresponse assumption in 11-1B, independence and repeated-test framing in 11-2B, and the workforce-level hypotheses and what a single chi-square says in 11-3A, which were still being revised when the session ended.",
    "No unstuck or release-valve use occurred in this session. The student's questions about tuples, arrays, pd.concat, and DataFrame.append were answered as free concept explanations, and the 'done' on 11-1A was accepted as a clear step rather than an escape-hatch use. One tutor error is worth noting: the first export of this conversation gave the session date as 2026-11-15 instead of 2026-11-16, and this second export replaces it."
  ]
}

## Before you submit

1. Keep `LE11.ipynb` next to a `data` folder containing the supplied `LE11_employee_commutes.csv`. Extract the complete student ZIP to preserve this arrangement.
2. **Restart Kernel and Run All Cells** after completing the notebook and confirm every cell runs without errors.
3. Confirm your calculated tables, printed values, and written answers are visible in the saved notebook.
4. Confirm your session record blocks are pasted into the cell above.
5. Save the notebook, then upload the `.ipynb` to Canvas, as directed.